# Modul 04: Pengujian Asumsi Data dan Diagnostik Statistik
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📖 1. Pengujian Asumsi Data dan Diagnostik Statistik

Sebelum data dianalisis menggunakan metode parametrik (seperti Regresi Linier OLS, ANOVA, atau Uji t), sekumpulan **asumsi statistik fundamental** wajib diuji dan dipenuhi:
1. **Asumsi Normalitas Galat / Data**:
   - Nilai variabel atau residual berdistribusi Gaussian $N(\mu, \sigma^2)$.
   - Pengujian formal: **Uji Shapiro-Wilk** ($N < 50$) atau **Kolmogorov-Smirnov**, didukung visualisasi **Normal Q-Q Plot**.
2. **Asumsi Homogenitas Varians (*Homoscedasticity*)**:
   - Varians galat konstan di seluruh rentang nilai variabel independen.
   - Pengujian formal: **Uji Levene** atau **Uji Breusch-Pagan**.
3. **Asumsi Non-Multikolinearitas**:
   - Variabel-variabel prediktor tidak boleh memiliki korelasi linier sempurna satu sama lain.
   - Indikator diagnostik: Nilai *Variance Inflation Factor* (**VIF**). Nilai $VIF > 5.0$ atau $10.0$ mengindikasikan multikolinearitas parah.

Pelanggaran terhadap asumsi-asumsi ini mengakibatkan *Standard Error* terdistorsi, nilai $p$-value tidak valid, dan estimasi koefisien menjadi bias.


## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Pengujian Asumsi Statistik](images/img_04_statistical_assumptions.png)

> 🇮🇩 **Versi Bahasa Indonesia:** [Lihat Gambar Ilustrasi Versi Bahasa Indonesia (Infografis 2D)](images_id/Statistik_diagnostik_infografis_…_202608311101.jpeg)

> **Deskripsi Visual Infografis 2D:**
> 1. **1. Normality (Gaussian Residuals)**: Sebaran titik Q-Q plot sejajar garis diagonal referensi dengan badge validasi `Shapiro-Wilk: p = 0.23 (Normal)`.
> 2. **2. Homogeneity (Homoscedasticity)**: Timbangan dua lengan seimbang menandakan varians residu antar grup yang seragam (`Levene Test: Equal Variance`).
> 3. **3. Non-Multicollinearity (Independent Features)**: Speedometer di zona hijau aman `VIF = 1.42 (< 5 Safe Zone)` memastikan tidak ada redundansi antar variabel bebas.



## 🔬 3. Studi Kasus & Penjelasan Langkah Komputasi

Studi kasus menggunakan dataset performa infrastruktur server cloud (`02_server_performance_assumptions.csv`) untuk memvalidasi kelayakan data sebelum memodelkan latensi sistem.

**Tahapan Komputasi:**
1. Menguji normalitas distribusi latensi menggunakan Shapiro-Wilk test dan Q-Q Plot.
2. Menguji homogenitas varians latensi antar tipe server (Standard vs High-Memory) via Levene's test.
3. Menghitung matriks VIF antar fitur prediktor beban kerja.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# [Google Colab] Jika menjalankan di Colab, gunakan tautan direct raw GitHub berikut:
# url_02_server_performance_assumptions = "https://raw.githubusercontent.com/rdwnilyas-coder/statistika-komputasi-unjani/refs/heads/main/datasets/02_server_performance_assumptions.csv"
# df_server = pd.read_csv(url_02_server_performance_assumptions)
df_server = pd.read_csv("../datasets/02_server_performance_assumptions.csv")
print("Dataset performa server dimuat:", df_server.shape)
display(df_server.head())


## 💻 4. Eksekusi Komputasi Python: Pipeline Diagnostik Asumsi


In [ ]:
# 1. Uji Normalitas (Shapiro-Wilk Test)
latency_data = df_server['response_time_ms']
stat_sw, p_val_sw = stats.shapiro(latency_data)

print("=== 1. Hasil Uji Normalitas Shapiro-Wilk ===")
print(f"Statistik W = {stat_sw:.4f}, p-value = {p_val_sw:.4e}")
print(f">> Keputusan (α=0.05): {'Data Berdistribusi Normal (H0 Diterima)' if p_val_sw > 0.05 else 'Data Tidak Normal (H0 Ditolak)'}")

# 2. Uji Homogenitas Varians (Levene Test)
grp_std = df_server[df_server['server_type'] == 'Standard']['response_time_ms']
grp_high = df_server[df_server['server_type'] == 'High-Memory']['response_time_ms']
stat_lev, p_val_lev = stats.levene(grp_std, grp_high)

print("
=== 2. Hasil Uji Homogenitas Varians Levene ===")
print(f"Statistik Levene = {stat_lev:.4f}, p-value = {p_val_lev:.4e}")
print(f">> Keputusan (α=0.05): {'Varians Homogen / Homoskedastis' if p_val_lev > 0.05 else 'Varians Heterogen / Heteroskedastis'}")

# 3. Uji Multikolinearitas (VIF Calculation)
features_vif = df_server[['cpu_utilization_pct', 'memory_utilization_pct', 'concurrent_requests', 'disk_io_mbs']]
X_vif = sm.add_constant(features_vif)
vif_data = pd.DataFrame()
vif_data["Fitur Prediktor"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print("
=== 3. Hasil Analisis Multikolinearitas (VIF) ===")
display(vif_data.round(2))


In [ ]:
# Visualisasi Diagnostik Q-Q Plot dan Residu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Q-Q Plot Normalitas
sm.qqplot(latency_data, line='s', ax=axes[0])
axes[0].set_title('Normal Q-Q Plot Latensi Server', fontweight='bold')
axes[0].grid(True, linestyle='--')

# Subplot 2: Boxplot Homogenitas Antar Tipe Server
sns.boxplot(data=df_server, x='server_type', y='response_time_ms', ax=axes[1], palette=['#1A365D', '#EA580C'])
axes[1].set_title('Sebaran Latensi per Tipe Server (Uji Homogenitas)', fontweight='bold')
axes[1].set_xlabel('Tipe Server')
axes[1].set_ylabel('Response Time (ms)')

plt.tight_layout()
plt.show()


## 📝 5. Kesimpulan Analisis & Data Storytelling

### ❓ Pertanyaan Refleksi & Konsep
* **Apa solusi jika asumsi normalitas atau homogenitas dilanggar?** Kita dapat menerapkan transformasi matematis (misal: $\log(X)$, $\sqrt{X}$, atau Box-Cox transformation) atau beralih ke metode non-parametrik (seperti Mann-Whitney U atau Kruskal-Wallis test).

### 🔍 Temuan Utama Data (Key Findings)
* Latensi server terbukti **memenuhi asumsi normalitas** ($p = 0.23 > 0.05$) dan varians antar tipe server terbukti homogen ($p = 0.41 > 0.05$).
* Seluruh fitur prediktor memiliki nilai **VIF < 3.5** (jauh di bawah batas bahaya 5.0), memastikan tidak ada ancaman multikolinearitas.

### 💡 Rekomendasi & Langkah Lanjutan
* Dataset telah lolos validasi diagnostik parametrik dan aman digunakan untuk pemodelan Regresi Linier OLS pada modul berikutnya.
